In [ ]:
import subprocess
subprocess.run(["pip", "install", "numpy==1.26.4", "--force-reinstall", "-q"], check=True)

from google.colab import drive
drive.mount('/content/drive')

# ✅ Only install what the demo actually needs — no datasets
!pip install -q gradio librosa soundfile torchaudio transformers pyctcdecode
!pip install -q https://github.com/kpu/kenlm/archive/master.zip
!apt-get install -y -qq build-essential cmake libboost-all-dev zlib1g-dev libbz2-dev liblzma-dev

import torch, os
print("=== INFO ===")
print(f"🖥️  Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

BASE_DIR       = "/content/drive/MyDrive/Arabic_ASR"
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoints"
LOG_DIR        = f"{BASE_DIR}/logs"
DATA_DIR       = f"{BASE_DIR}/data"

for d in [BASE_DIR, CHECKPOINT_DIR, LOG_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"📁 {d}")

import librosa, numpy as np
print(f"✅ numpy version: {np.__version__}")
print("\n✅ CELL 1 COMPLETE — Run Cell 4 (skip 2 and 3)")

In [ ]:
import numpy as np
import torch, librosa, os, pickle, warnings, random
warnings.filterwarnings('ignore')
from datasets import load_dataset, Audio
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# ── Config ────────────────────────────────────────────────────
MAX_SAMPLES  = 10000
MAX_DURATION = 7.0
MIN_DURATION = 0.5
SAMPLE_RATE  = 16000
N_MFCC       = 40
FEAT_DIM     = 120        # 40 mfcc + 40 delta + 40 delta-delta
MAX_MFCC_LEN = 400
BATCH_SIZE   = 32
CACHE_FILE   = f"{DATA_DIR}/processed_10k_delta.pkl"   # new cache name so it rebuilds

# ── Vocab ─────────────────────────────────────────────────────
ARABIC_CHARS = list("ابتثجحخدذرزسشصضطظعغفقكلمنهوي ءأإآؤئة")
BLANK_IDX    = 0
CHAR2IDX     = {c: i+1 for i, c in enumerate(ARABIC_CHARS)}
IDX2CHAR     = {i+1: c  for i, c in enumerate(ARABIC_CHARS)}
IDX2CHAR[0]  = ""
VOCAB_SIZE   = len(ARABIC_CHARS) + 1
print(f"📖 Vocab size: {VOCAB_SIZE}")
print(f"📐 Feature dim: {FEAT_DIM}  (MFCC + Δ + ΔΔ)")

# ── Feature extraction: MFCC + delta + delta-delta ────────────
def extract_mfcc(audio, sr=SAMPLE_RATE):
    audio = audio.astype(np.float32)
    if np.abs(audio).max() > 1.0:
        audio = audio / 32768.0
    rms = np.sqrt(np.mean(audio**2))
    if rms > 0:
        audio = audio / (rms + 1e-8) * 0.1

    mfcc   = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC, hop_length=256, n_fft=512)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat   = np.vstack([mfcc, delta, delta2])          # (120, T)

    feat = (feat - feat.mean()) / (feat.std() + 1e-8)

    T        = feat.shape[1]
    real_len = min(T, MAX_MFCC_LEN)

    if T < MAX_MFCC_LEN:
        feat = np.pad(feat, ((0,0),(0, MAX_MFCC_LEN - T)), mode='constant')
    else:
        feat = feat[:, :MAX_MFCC_LEN]

    return feat.T.astype(np.float16), real_len          # (400, 120)

def text_to_indices(text):
    return [CHAR2IDX[c] for c in text if c in CHAR2IDX]

# ── Load / Cache ───────────────────────────────────────────────
if os.path.exists(CACHE_FILE):
    print(f"📦 Loading from cache: {CACHE_FILE}")
    with open(CACHE_FILE, 'rb') as f:
        data = pickle.load(f)
    print(f"✅ {len(data)} samples loaded")
else:
    print("⬇️  Downloading dataset...")
    raw = load_dataset("MAdel121/arabic-egy-cleaned", split="train")
    raw = raw.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

    data, skipped = [], 0
    for sample in raw:
        if len(data) >= MAX_SAMPLES:
            break
        try:
            arr = np.array(sample['audio']['array'], dtype=np.float32)
            sr  = sample['audio']['sampling_rate']
            dur = len(arr) / sr

            if not (MIN_DURATION <= dur <= MAX_DURATION):
                skipped += 1; continue

            text    = str(sample['text']).strip()
            indices = text_to_indices(text)

            if len(text) < 2 or len(indices) < 2:
                skipped += 1; continue

            feat, real_len = extract_mfcc(arr, sr)

            data.append({
                'mfcc'    : feat,
                'label'   : indices,
                'text'    : text,
                'real_len': real_len,
            })

            if len(data) % 1000 == 0:
                print(f"  ✅ {len(data)}/{MAX_SAMPLES}")

        except:
            skipped += 1

    print(f"\n✅ Kept: {len(data)} | Skipped: {skipped}")
    ram_mb = len(data) * MAX_MFCC_LEN * FEAT_DIM * 2 / 1e6
    print(f"💾 Estimated RAM: {ram_mb:.0f} MB")

    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(data, f)
    print(f"💾 Cached → {CACHE_FILE}")

# ── Split ──────────────────────────────────────────────────────
train_data, val_data = train_test_split(data, test_size=0.1, random_state=42)
print(f"📊 Train: {len(train_data)} | Val: {len(val_data)}")

# ── Dataset & DataLoader ───────────────────────────────────────
class ArabicASRDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s    = self.samples[idx]
        mfcc = torch.tensor(s['mfcc'].astype(np.float32), dtype=torch.float32)
        return mfcc, torch.tensor(s['label'], dtype=torch.long), s['real_len']

def collate_fn(batch):
    mfccs, labels, real_lens = zip(*batch)
    return (
        torch.stack(mfccs),
        torch.cat(labels),
        torch.tensor(real_lens, dtype=torch.long),
        torch.tensor([len(l) for l in labels], dtype=torch.long),
    )

train_loader = DataLoader(ArabicASRDataset(train_data), batch_size=BATCH_SIZE,
    shuffle=True,  collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(ArabicASRDataset(val_data),   batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f"✅ Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print(f"✅ Feature dim  : {FEAT_DIM}")
print("\n✅ CELL 2 COMPLETE — Run Cell 3")

In [ ]:
import torch, torch.nn as nn, time, jiwer, random, os
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {DEVICE}")

BEST_MODEL  = f"{CHECKPOINT_DIR}/best_model.pt"
RESUME_CKPT = f"{CHECKPOINT_DIR}/resume_checkpoint.pt"
LOG_FILE    = f"{LOG_DIR}/training_log.txt"

# ── SpecAugment ───────────────────────────────────────────────
def spec_augment(mfcc, freq_mask=20, time_mask=40, n_freq=2, n_time=2):
    mfcc = mfcc.clone()
    T, F = mfcc.shape
    for _ in range(n_freq):
        f  = random.randint(1, freq_mask)
        f0 = random.randint(0, max(1, F - f))
        mfcc[:, f0:f0+f] = 0.0
    for _ in range(n_time):
        t  = random.randint(1, time_mask)
        t0 = random.randint(0, max(1, T - t))
        mfcc[t0:t0+t, :] = 0.0
    return mfcc

# ── Early Stopping — watches CER, patience=15 ─────────────────
class EarlyStopping:
    def __init__(self, patience=7, min_delta=1e-3):
        self.patience    = patience
        self.min_delta   = min_delta
        self.counter     = 0
        self.best        = float('inf')
        self.should_stop = False
    def step(self, cer):
        if cer < self.best - self.min_delta:
            self.best    = cer
            self.counter = 0
        else:
            self.counter += 1
            print(f"       ⏳ No CER improvement {self.counter}/{self.patience} (best={self.best*100:.1f}%)")
            if self.counter >= self.patience:
                self.should_stop = True

# ── Model ──────────────────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch), nn.GELU(),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
        )
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(x + self.net(x))

class CNNBiLSTM(nn.Module):
    def __init__(self, feat_dim=120, vocab_size=38, hidden=320, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.MaxPool2d((2,1)),
            nn.Dropout2d(0.05),
            ResBlock(32),
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.GELU(),
            nn.MaxPool2d((2,1)),
            nn.Dropout2d(0.05),
        )
        cnn_out   = (feat_dim // 4) * 64
        self.proj = nn.Sequential(
            nn.Linear(cnn_out, hidden, bias=False),
            nn.LayerNorm(hidden), nn.GELU(),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(hidden, hidden, num_layers=3,
            batch_first=True, bidirectional=True, dropout=dropout)
        self.ln   = nn.LayerNorm(hidden * 2)
        self.head = nn.Linear(hidden * 2, vocab_size)

    def forward(self, x):
        B, T, F = x.shape
        x = x.permute(0,2,1).unsqueeze(1)
        x = self.cnn(x)
        _, C, F2, T2 = x.shape
        x = x.permute(0,3,1,2).reshape(B, T2, C*F2)
        x = self.proj(x)
        x, _ = self.lstm(x)
        x = self.ln(x)
        x = self.head(x)
        return torch.log_softmax(x.permute(1,0,2), dim=-1)

# ── Init ───────────────────────────────────────────────────────
model     = CNNBiLSTM(feat_dim=FEAT_DIM, vocab_size=VOCAB_SIZE, hidden=320, dropout=0.2).to(DEVICE)
ctc_loss  = nn.CTCLoss(blank=BLANK_IDX, reduction='mean', zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
EPOCHS    = 80

plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)

scaler         = torch.cuda.amp.GradScaler()
early_stopping = EarlyStopping(patience=15)

# ── Resume ─────────────────────────────────────────────────────
start_epoch = 1
best_val    = float('inf')
best_cer    = float('inf')
log_lines   = []

if os.path.exists(RESUME_CKPT):
    print(f"🔄 Resuming from checkpoint...")
    ckpt = torch.load(RESUME_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scaler.load_state_dict(ckpt['scaler_state'])
    # reset early stopping with new patience/metric — don't restore old state
    start_epoch = ckpt['epoch'] + 1
    best_val    = ckpt['best_val']
    best_cer    = ckpt.get('best_cer', float('inf'))
    early_stopping.best = best_cer   # start from known best CER
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r') as f:
            log_lines = f.read().splitlines()
    print(f"✅ Resumed from epoch {ckpt['epoch']}")
    print(f"   best_val={best_val:.4f}  best_cer={best_cer*100:.1f}%")
    print(f"   continuing with patience=15 watching CER")
else:
    print("🆕 Starting fresh")

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Params: {total:,}")

# ── Decoders ───────────────────────────────────────────────────
def ctc_greedy_decode(log_probs):
    indices = log_probs.argmax(dim=-1).permute(1, 0)
    out = []
    for seq in indices:
        dec, prev = [], -1
        for idx in seq.tolist():
            if idx != prev and idx != BLANK_IDX:
                dec.append(idx)
            prev = idx
        out.append(dec)
    return out

def indices_to_text(idx):
    return "".join(IDX2CHAR.get(i, "") for i in idx)

# ── Metrics ────────────────────────────────────────────────────
def compute_metrics(model, val_data, max_samples=200):
    model.eval()
    refs, hyps = [], []
    with torch.no_grad():
        for s in val_data[:max_samples]:
            mfcc = torch.tensor(s['mfcc'], dtype=torch.float32).unsqueeze(0).to(DEVICE)
            lp   = model(mfcc)
            pred = indices_to_text(ctc_greedy_decode(lp)[0])
            refs.append(s['text'])
            hyps.append(pred if pred.strip() else " ")
    try:
        wer = jiwer.wer(refs, hyps)
        cer = jiwer.cer(refs, hyps)
    except:
        wer, cer = 1.0, 1.0
    return wer, cer

# ── Train / Val ────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total, count = 0.0, 0
    for mfccs, labels, mfcc_lens, label_lens in loader:
        aug    = torch.stack([spec_augment(m) for m in mfccs]).to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            lp   = model(aug)
            loss = ctc_loss(lp, labels, mfcc_lens.to(DEVICE), label_lens.to(DEVICE))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item(); count += 1
    return total / count

def validate(model, loader):
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for mfccs, labels, mfcc_lens, label_lens in loader:
            mfccs  = mfccs.to(DEVICE)
            labels = labels.to(DEVICE)
            with torch.cuda.amp.autocast():
                lp   = model(mfccs)
                loss = ctc_loss(lp, labels, mfcc_lens.to(DEVICE), label_lens.to(DEVICE))
            total += loss.item(); count += 1
    return total / count

# ── Training Loop ──────────────────────────────────────────────
print(f"\n🚀 Continuing from epoch {start_epoch} → {EPOCHS}\n")
print(f"{'Ep':>4}  {'TrainL':>7}  {'ValL':>7}  {'WER':>7}  {'CER':>7}  {'LR':>9}  {'Time':>5}")
print("─" * 70)

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
    val_loss   = validate(model, val_loader)

    prev_lr = optimizer.param_groups[0]['lr']
    plateau_scheduler.step(val_loss)
    cur_lr  = optimizer.param_groups[0]['lr']
    if cur_lr != prev_lr:
        print(f"📉 LR reduced: {prev_lr:.2e} → {cur_lr:.2e}")

    wer, cer = compute_metrics(model, val_data, max_samples=200)
    elapsed  = time.time() - t0

    model.eval()
    with torch.no_grad():
        mfcc = torch.tensor(val_data[0]['mfcc'], dtype=torch.float32).unsqueeze(0).to(DEVICE)
        pred = indices_to_text(ctc_greedy_decode(model(mfcc))[0])

    t_s = val_data[0]['text'][:40]
    p_s = pred[:40] if pred.strip() else "(empty)"

    row = (f"{epoch:>4}  {train_loss:>7.4f}  {val_loss:>7.4f}  "
           f"{wer*100:>6.1f}%  {cer*100:>6.1f}%  {cur_lr:>9.2e}  {elapsed:>4.0f}s")
    print(row)
    print(f"       🎯 {t_s}")
    print(f"       🤖 {p_s}\n")
    log_lines.append(row + f" | {t_s} | {p_s}")

    # ── Save best model (tracked by CER now) ──────────────────
    if cer < best_cer:
        best_cer = cer
        best_val = val_loss
        torch.save({
            'epoch'      : epoch,
            'model_state': model.state_dict(),
            'val_loss'   : val_loss,
            'wer'        : wer,
            'cer'        : cer,
            'vocab_size' : VOCAB_SIZE,
            'char2idx'   : CHAR2IDX,
            'idx2char'   : IDX2CHAR,
            'feat_dim'   : FEAT_DIM,
            'n_mfcc'     : N_MFCC,
            'max_len'    : MAX_MFCC_LEN,
        }, BEST_MODEL)
        print(f"       💾 Best model saved! CER={cer*100:.1f}%  WER={wer*100:.1f}%  val={val_loss:.4f}\n")

    # ── Save resume checkpoint ─────────────────────────────────
    torch.save({
        'epoch'          : epoch,
        'model_state'    : model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scaler_state'   : scaler.state_dict(),
        'es_counter'     : early_stopping.counter,
        'es_best'        : early_stopping.best,
        'es_should_stop' : early_stopping.should_stop,
        'best_val'       : best_val,
        'best_cer'       : best_cer,
    }, RESUME_CKPT)

    # ── Save log ───────────────────────────────────────────────
    with open(LOG_FILE, 'w') as f:
        f.write("\n".join(log_lines))

    # ── Early stop on CER ──────────────────────────────────────
    early_stopping.step(cer)
    if early_stopping.should_stop:
        print(f"🛑 Early stop at epoch {epoch} — CER not improving for {early_stopping.patience} epochs")
        break

print(f"\n✅ Done!")
print(f"   Best CER : {best_cer*100:.1f}%")
print(f"   Best WER : loaded from best_model.pt")
print(f"💾 Best model  → {BEST_MODEL}")
print(f"💾 Resume ckpt → {RESUME_CKPT}")
print("\n✅ CELL 3 COMPLETE — Run Cell 4")

In [ ]:
LOG_FILE = f"{LOG_DIR}/training_log.txt"

if os.path.exists(LOG_FILE):
    print("📋 Full Training History\n")
    print(f"{'Ep':>4}  {'TrainL':>7}  {'ValL':>7}  {'WER':>7}  {'CER':>7}  {'LR':>9}  {'Time':>5}")
    print("─" * 70)
    with open(LOG_FILE, 'r') as f:
        for line in f.read().splitlines():
            print(line)
    print("\n✅ Training complete — best model was saved at epoch 37  CER=35.1%")
else:
    print("⚠️ No log file found")

In [ ]:
import gradio as gr
gr.close_all()

In [ ]:
# ── Numpy fix first ────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "numpy==1.26.4", "--force-reinstall", "-q"])

# ── Install ────────────────────────────────────────────────────
!apt-get update -qq
!apt-get install -y -qq build-essential cmake libboost-all-dev zlib1g-dev libbz2-dev liblzma-dev
!pip install -q https://github.com/kpu/kenlm/archive/master.zip
!pip install -q pyctcdecode gradio torchaudio

# ── Force fresh reimport ───────────────────────────────────────
import sys
for mod in list(sys.modules.keys()):
    if 'kenlm' in mod or 'pyctcdecode' in mod:
        del sys.modules[mod]

import kenlm
from pyctcdecode import build_ctcdecoder
print(f"✅ kenlm loaded")

import os, re, collections, torch, librosa, numpy as np, gradio as gr
import torch.nn as nn
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {DEVICE}")

# ── Paths ──────────────────────────────────────────────────────
BASE_DIR       = "/content/drive/MyDrive/Arabic_ASR"
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoints"
LOG_DIR        = f"{BASE_DIR}/logs"
DATA_DIR       = f"{BASE_DIR}/data"
BEST_MODEL     = f"{CHECKPOINT_DIR}/best_model.pt"
LM_TEXT        = f"{DATA_DIR}/lm_corpus.txt"
ARPA_FILE      = f"{DATA_DIR}/arabic_lm.arpa"
BINARY_FILE    = f"{DATA_DIR}/arabic_lm.binary"
LMPLZ          = "/content/kenlm/build/bin/lmplz"
BUILD_BINARY   = "/content/kenlm/build/bin/build_binary"

# ── Config ─────────────────────────────────────────────────────
SAMPLE_RATE  = 16000
N_MFCC       = 40
MAX_MFCC_LEN = 400
FEAT_DIM     = 120

# ── Vocab ──────────────────────────────────────────────────────
ARABIC_CHARS = list("ابتثجحخدذرزسشصضطظعغفقكلمنهوي ءأإآؤئة")
BLANK_IDX    = 0
CHAR2IDX     = {c: i+1 for i, c in enumerate(ARABIC_CHARS)}
IDX2CHAR     = {i+1: c  for i, c in enumerate(ARABIC_CHARS)}
IDX2CHAR[0]  = ""
VOCAB_SIZE   = len(ARABIC_CHARS) + 1

# ── Arabic normalization ───────────────────────────────────────
def normalize_ar(text):
    text = text.strip()
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ة", "ه").replace("ى", "ي")
    text = re.sub(r"\s+", " ", text)
    return text

# ── Model definition ───────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch), nn.GELU(),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
        )
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(x + self.net(x))

class CNNBiLSTM(nn.Module):
    def __init__(self, feat_dim=120, vocab_size=38, hidden=320, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.MaxPool2d((2,1)),
            nn.Dropout2d(0.05),
            ResBlock(32),
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.GELU(),
            nn.MaxPool2d((2,1)),
            nn.Dropout2d(0.05),
        )
        cnn_out   = (feat_dim // 4) * 64
        self.proj = nn.Sequential(
            nn.Linear(cnn_out, hidden, bias=False),
            nn.LayerNorm(hidden), nn.GELU(),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(hidden, hidden, num_layers=3,
            batch_first=True, bidirectional=True, dropout=dropout)
        self.ln   = nn.LayerNorm(hidden * 2)
        self.head = nn.Linear(hidden * 2, vocab_size)

    def forward(self, x):
        B, T, F = x.shape
        x = x.permute(0,2,1).unsqueeze(1)
        x = self.cnn(x)
        _, C, F2, T2 = x.shape
        x = x.permute(0,3,1,2).reshape(B, T2, C*F2)
        x = self.proj(x)
        x, _ = self.lstm(x)
        x = self.ln(x)
        x = self.head(x)
        return torch.log_softmax(x.permute(1,0,2), dim=-1)

# ── Build KenLM (skipped if binary already exists) ────────────
if not os.path.exists(BINARY_FILE):
    if not os.path.exists("/content/kenlm"):
        !git clone https://github.com/kpu/kenlm.git /content/kenlm
    !mkdir -p /content/kenlm/build
    !cd /content/kenlm/build && cmake .. && make -j4
    assert os.path.exists(LMPLZ), "❌ lmplz not found"

    print("📝 Building LM corpus...")
    with open(LM_TEXT, "w", encoding="utf-8") as f:
        for s in tqdm(data):
            text = normalize_ar(s['text'])
            if len(text.split()) >= 2:
                f.write(text + "\n")

    print("🧠 Training 5-gram LM...")
    !{LMPLZ} -o 5 < "{LM_TEXT}" > "{ARPA_FILE}"

    print("⚡ Binarizing...")
    !{BUILD_BINARY} "{ARPA_FILE}" "{BINARY_FILE}"
    print(f"✅ LM binary → {BINARY_FILE}")
else:
    print(f"📦 LM binary already exists → {BINARY_FILE}")

# ── Unigrams ───────────────────────────────────────────────────
unigrams = []
if os.path.exists(LM_TEXT):
    counts = collections.defaultdict(int)
    with open(LM_TEXT, "r", encoding="utf-8") as f:
        for line in f:
            for word in line.strip().split():
                counts[word] += 1
    unigrams = list(counts.keys())
    print(f"✅ {len(unigrams)} unigrams")

# ── Load best model ────────────────────────────────────────────
checkpoint = torch.load(BEST_MODEL, map_location=DEVICE)
model = CNNBiLSTM(
    feat_dim=checkpoint['feat_dim'],
    vocab_size=checkpoint['vocab_size'],
    hidden=320, dropout=0.2
).to(DEVICE)
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f"✅ Model loaded — epoch {checkpoint['epoch']}  WER={checkpoint['wer']*100:.1f}%  CER={checkpoint['cer']*100:.1f}%")

# ── Build KenLM beam decoder ───────────────────────────────────
labels            = [IDX2CHAR[i] for i in range(VOCAB_SIZE)]
labels[BLANK_IDX] = ""

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=BINARY_FILE,
    unigrams=unigrams if unigrams else None,
)
print("✅ KenLM beam decoder ready")

# ── Greedy decoder ─────────────────────────────────────────────
def ctc_greedy_decode_single(log_probs):
    indices = log_probs.argmax(dim=-1).squeeze(1)
    dec, prev = [], -1
    for idx in indices.tolist():
        if idx != prev and idx != BLANK_IDX:
            dec.append(idx)
        prev = idx
    return "".join(IDX2CHAR.get(i, "") for i in dec)

# ── Feature extraction ─────────────────────────────────────────
def extract_features(audio_path):
    audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    audio = audio.astype(np.float32)

    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak

    audio = np.append(audio[0], audio[1:] - 0.97 * audio[:-1])

    try:
        audio, _ = librosa.effects.trim(audio, top_db=25)
    except:
        pass

    min_len = int(sr * 0.5)
    if len(audio) < min_len:
        audio = np.pad(audio, (0, min_len - len(audio)))

    rms = np.sqrt(np.mean(audio**2))
    if rms > 0:
        audio = audio / (rms + 1e-8) * 0.1

    mfcc   = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC,
                                   hop_length=256, n_fft=512)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat   = np.vstack([mfcc, delta, delta2])

    feat = (feat - feat.mean()) / (feat.std() + 1e-8)

    T = feat.shape[1]
    if T < MAX_MFCC_LEN:
        feat = np.pad(feat, ((0,0),(0, MAX_MFCC_LEN - T)), mode='constant')
    else:
        feat = feat[:, :MAX_MFCC_LEN]

    return feat.T.astype(np.float32)

# ── Transcribe ─────────────────────────────────────────────────
def transcribe(audio_path):
    if audio_path is None:
        return "⚠️ No audio uploaded."
    try:
        feat = extract_features(audio_path)
        x    = torch.tensor(feat).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            log_probs = model(x)

        probs       = torch.exp(log_probs).permute(1,0,2).cpu().numpy()[0]
        beam_text   = normalize_ar(decoder.decode(probs)).strip()
        greedy_text = normalize_ar(ctc_greedy_decode_single(log_probs)).strip()

        result = beam_text if len(beam_text) >= len(greedy_text) else greedy_text
        return result if result else "(empty — speak louder or closer to mic)"

    except Exception as e:
        return f"❌ Error: {str(e)}"

# ── Gradio UI ──────────────────────────────────────────────────
demo = gr.Interface(
    fn=transcribe,
    inputs=gr.Audio(
        type="filepath",
        label="🎤 Upload or Record Arabic Speech",
    ),
    outputs=gr.Textbox(
        label="📝 Transcription",
        rtl=True,
        show_copy_button=True,
    ),
    title="🇪🇬 Arabic Egyptian ASR — CNN+BiLSTM+CTC",
    description="""
Record or upload Egyptian Arabic speech.
Uses CNN + Residual Block + BiLSTM + CTC with KenLM 5-gram beam search.

**Tips for best results:**
- Speak clearly at normal pace
- Keep under 7 seconds
- Upload phone recordings for best quality
- Egyptian Arabic dialect only
""",
    allow_flagging="never",
)

demo.launch(inline=True, share=False, debug=False, quiet=False)

In [ ]:
# ── Test on 10 validation samples to confirm model works ──────
model.eval()
print("Testing model on clean validation data:\n")
for i in range(10):
    s = val_data[i]
    mfcc = torch.tensor(s['mfcc'], dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        log_probs = model(mfcc)

    # Beam decode
    probs     = torch.exp(log_probs).permute(1,0,2).cpu().numpy()[0]
    beam_text = decoder.decode(probs)
    beam_text = normalize_ar(beam_text).strip()

    print(f"Sample {i+1}")
    print(f"  🎯 REF : {s['text']}")
    print(f"  🤖 PRED: {beam_text if beam_text else '(empty)'}")
    print()